# Training della value network su Kaggle (GPU, dataset completo)

Gemello di `train_colab.ipynb` per Kaggle Notebooks. La differenza che conta:
Kaggle offre **~29 GB di RAM** anche con GPU, quindi il dataset completo
(~1,58M posizioni, ~20 GB una volta caricato) ci sta e si allena con
`--sample 1.0` invece del 40% a cui costringe il Colab gratuito.
Produce **solo il checkpoint dei pesi**: l'export TorchScript per il C++ va
fatto in locale (`torch_geometric==2.6.1`), vedi `docs/spiegazione_value_network.md` §6.

### Prima di eseguire

1. **Account Kaggle verificato** (numero di telefono): serve per GPU e Internet.
2. **Crea un Dataset Kaggle privato** (*Datasets → New Dataset*) caricando i tre
   file della cartella Drive `HiveGotThis_colab/` (o i loro equivalenti locali):
   `hive_value_gnn.py`, `train_hive_value_gnn.py`, `dataset_jsonl.zip`.
   NB: Kaggle **scompatta da solo** gli zip caricati — nel Dataset compariranno
   direttamente i `boardspace_*.jsonl`, ed è quello che le celle si aspettano.
3. **Nuovo Notebook** (*Code → New Notebook*), poi importa questo file
   (*File → Import Notebook*) e nel pannello di destra:
   - *Input → Add Input*: collega il dataset creato al passo 2;
   - *Session options → Accelerator*: **GPU T4 x2** (o P100);
   - *Session options → Internet*: **ON** (serve per `pip install`).
4. Esegui le celle in ordine. Per un run non presidiato usa *Save Version →
   Save & Run All*: esegue tutto in background e conserva l'output anche se
   il browser si chiude (la sessione interattiva invece muore con lui).

Tempi indicativi su T4: ~25-30 minuti di caricamento dati + ~3 minuti a epoca
(~2 ore e mezza per 30 epoche). La quota gratuita e' 30 ore di GPU a settimana.

In [ ]:
# Trova script e dati nel Dataset collegato come Input, a qualunque
# profondita'. NB: Kaggle scompatta da solo gli zip caricati in un Dataset,
# quindi di norma i .jsonl sono gia' estratti (e lo zip non esiste piu').
import glob

def find_one(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    return hits[0] if hits else None

TRAIN_PY = find_one('train_hive_value_gnn.py')
MODEL_PY = find_one('hive_value_gnn.py')
assert TRAIN_PY and MODEL_PY, 'script .py non trovati: collega il Dataset dal pannello Input'

INPUT_JSONL = sorted(glob.glob('/kaggle/input/**/*.jsonl', recursive=True))
ZIP = find_one('dataset_jsonl.zip')
assert INPUT_JSONL or ZIP, 'nessun .jsonl e nessun dataset_jsonl.zip nel Dataset collegato'

CHECKPOINT = '/kaggle/working/checkpoint_gen0_full.pt'
print(f"{len(INPUT_JSONL)} jsonl gia' estratti" if INPUT_JSONL else f'zip da scompattare: {ZIP}')

In [ ]:
# Verifica GPU e installa PyTorch Geometric.
# NB: qui la versione di torch_geometric e' libera perche' si fa solo training;
# il pin ==2.6.1 riguarda solo l'export TorchScript, che si fa in locale.
import torch
print('GPU disponibile:', torch.cuda.is_available(),
      '-', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'attiva la GPU nelle Session options!')

%pip install -q torch_geometric

In [ ]:
# Copia gli script in /kaggle/working e prepara la lista dei file dati.
# Se i .jsonl sono gia' estratti nell'input si leggono da li' (read-only va
# bene); se invece c'e' ancora lo zip, si scompatta su disco effimero
# (/kaggle/tmp): NON in /kaggle/working, altrimenti i 12 GB di JSONL
# finirebbero salvati come output del notebook a ogni versione.
import subprocess

subprocess.run(['cp', TRAIN_PY, MODEL_PY, '/kaggle/working/'], check=True)

if INPUT_JSONL:
    DATA_FILES = INPUT_JSONL
else:
    !mkdir -p /kaggle/tmp/dataset && unzip -o -q "{ZIP}" -d /kaggle/tmp/dataset
    DATA_FILES = sorted(glob.glob('/kaggle/tmp/dataset/**/*.jsonl', recursive=True))

print(len(DATA_FILES), 'file jsonl:')
for f in DATA_FILES:
    print(' ', f)

In [ ]:
# Training sul dataset COMPLETO (--sample non serve: il default e' 1.0).
# In RAM servono ~20 GB sui ~29 disponibili: se la sessione morisse durante
# il caricamento, aggiungere --sample 0.7 al comando.
# python -u = output senza buffering: l'avanzamento (caricamento file per
# file, poi ~10 righe per epoca con MSE parziale e posizioni/s) compare in
# tempo reale. Il salvataggio e' best-su-validation: piu' epoche non fanno danni.
files = ' '.join(DATA_FILES)

!cd /kaggle/working && python -u train_hive_value_gnn.py {files} \
    --output "{CHECKPOINT}" \
    --device cuda --batch-size 256 --epochs 30

### Dopo il training

Il checkpoint migliore è in `/kaggle/working/checkpoint_gen0_full.pt`: si
scarica dal file browser del pannello di destra (sessione interattiva) o
dalla scheda *Output* del notebook (dopo *Save & Run All*). Conviene anche
caricarlo nella cartella Drive `HiveGotThis_colab/`, dove stanno gli altri
checkpoint. In locale:

```bash
# esporta per il C++ (ambiente locale con torch_geometric==2.6.1)
python3 scripts/export_hive_value_gnn.py --weights checkpoint_gen0_full.pt --output hive_value_gnn.pt

# il motore la usa
./build/HiveEngine hive_value_gnn.pt
```

Il confronto onesto con il checkpoint Colab (40% dei dati) passa
dall'harness: stesso match, stesse condizioni, un modello per lato
(`tools/uhp_match.py`, doc §8).